# 추가 데이터 시각화 예시

타이타닉과 자전거 수요 예측 데이터를 사용해 조금 더 흥미로운 시각화를 만들어 봅니다.

이 노트북의 목표는 모델 성능을 높이는 것이 아니라, 데이터를 더 설득력 있게 보여주는 방법을 익히는 것입니다. 마케팅, 기획, 데이터 분석 직무에서는 숫자를 계산하는 것만큼이나 “어떤 관점으로 보여줄 것인가”가 중요합니다.

사용하는 시각화는 다음과 같습니다.

- 인터랙티브 막대그래프
- 선버스트 차트
- 트리맵
- 움직이는 선 그래프
- 움직이는 버블 차트
- 히트맵
- 3D 산점도

## 1. 환경 설정

`plotly`는 마우스를 올려 값을 확인하거나, 범례를 클릭해 항목을 숨기거나, 애니메이션을 재생할 수 있는 시각화 라이브러리입니다.

In [1]:
# 이 셀은 데이터 처리와 인터랙티브 시각화에 필요한 라이브러리를 불러오기 위해 필요합니다.
from pathlib import Path

import pandas as pd
import plotly.express as px


In [2]:
# 이 셀은 Plotly 그래프의 기본 스타일과 색상 팔레트를 정하기 위해 필요합니다.
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set2

## 2. 데이터 불러오기

타이타닉 데이터와 자전거 수요 예측 데이터를 함께 사용합니다.

In [3]:
# 이 셀은 타이타닉 데이터와 자전거 수요 예측 데이터가 들어 있는 폴더 위치를 지정하기 위해 필요합니다.
TITANIC_DIR = Path("titanic")
BIKE_DIR = Path("자전거수요예측")

In [4]:
# 이 셀은 타이타닉 학습 데이터를 읽기 위해 필요합니다.
titanic = pd.read_csv(TITANIC_DIR / "train.csv")

In [5]:
# 이 셀은 자전거 수요 예측 학습 데이터를 읽기 위해 필요합니다.
bike = pd.read_csv(BIKE_DIR / "train.csv", parse_dates=["datetime"])

In [6]:
# 이 셀은 두 데이터의 크기를 간단히 확인하기 위해 필요합니다.
titanic.shape, bike.shape

((891, 12), (10886, 12))

## 3. 타이타닉 데이터 준비

시각화에서 바로 읽기 쉽도록 영어 코드값을 한국어 라벨로 바꿉니다.

### 시각화용 라벨 만들기

생존 여부, 객실 등급, 성별, 탑승 항구를 그래프에서 바로 읽기 쉬운 값으로 바꿉니다.


In [7]:
# 이 셀은 타이타닉 데이터에 시각화용 한국어 라벨을 추가하기 위해 필요합니다.
# 원본 데이터는 0, 1 또는 영어 코드로 되어 있어 그래프에서 바로 읽기 어렵습니다.

# 생존 여부 숫자를 한국어로 바꿉니다.
survival_label = {0: "사망", 1: "생존"}
titanic["생존여부"] = titanic["Survived"].map(survival_label)

# 객실 등급 숫자를 1등석, 2등석, 3등석으로 바꿉니다.
class_label = {1: "1등석", 2: "2등석", 3: "3등석"}
titanic["객실등급"] = titanic["Pclass"].map(class_label)

# 성별 영어 값을 한국어로 바꿉니다.
sex_label = {"male": "남성", "female": "여성"}
titanic["성별"] = titanic["Sex"].map(sex_label)

# 탑승 항구 코드를 항구 이름으로 바꿉니다.
embarked_label = {"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"}
titanic["탑승항구"] = titanic["Embarked"].map(embarked_label)

# Plotly에서 승객 수를 합산할 수 있도록 승객 한 명을 1로 표시합니다.
titanic["인원"] = 1

titanic[["생존여부", "객실등급", "성별", "탑승항구", "인원"]].head()

,생존여부,객실등급,성별,탑승항구,인원
0,사망,3등석,남성,Southampton,1
1,생존,1등석,여성,Cherbourg,1
2,생존,3등석,여성,Southampton,1
3,생존,1등석,여성,Southampton,1
4,사망,3등석,남성,Southampton,1


### 연령대 만들기

숫자로 된 나이를 몇 개의 연령대 그룹으로 묶어 세그먼트 비교에 사용할 수 있게 만듭니다.


In [8]:
# 이 셀은 나이를 구간으로 나누어 세대별 시각화에 활용하기 위해 필요합니다.
# 숫자로 된 나이는 그대로 보면 세그먼트 비교가 어렵습니다.
# 그래서 나이를 어린이, 청소년, 청년처럼 사람이 읽기 쉬운 그룹으로 바꿉니다.

# 먼저 모든 승객의 연령대를 "나이 미상"으로 넣어 둡니다.
# Age가 비어 있는 승객은 이 값이 그대로 남게 됩니다.
titanic["연령대"] = "나이 미상"

# 0세 초과 12세 이하를 어린이로 분류합니다.
child_mask = (titanic["Age"] > 0) & (titanic["Age"] <= 12)
titanic.loc[child_mask, "연령대"] = "어린이"

# 12세 초과 19세 이하를 청소년으로 분류합니다.
teen_mask = (titanic["Age"] > 12) & (titanic["Age"] <= 19)
titanic.loc[teen_mask, "연령대"] = "청소년"

# 19세 초과 35세 이하를 청년으로 분류합니다.
young_mask = (titanic["Age"] > 19) & (titanic["Age"] <= 35)
titanic.loc[young_mask, "연령대"] = "청년"

# 35세 초과 60세 이하를 중장년으로 분류합니다.
middle_mask = (titanic["Age"] > 35) & (titanic["Age"] <= 60)
titanic.loc[middle_mask, "연령대"] = "중장년"

# 60세 초과를 고령으로 분류합니다.
senior_mask = titanic["Age"] > 60
titanic.loc[senior_mask, "연령대"] = "고령"

titanic[["Age", "연령대"]].head(10)

,Age,연령대
0,22.0,청년
1,38.0,중장년
2,26.0,청년
3,35.0,청년
4,35.0,청년
5,NaN,나이 미상
6,54.0,중장년
7,2.0,어린이
8,27.0,청년
9,14.0,청소년


### 이름에서 호칭 추출하기

승객 이름에 들어 있는 Mr, Mrs, Miss 같은 호칭을 꺼내고 자주 쓰는 값 중심으로 정리합니다.


In [9]:
# 이 셀은 이름에서 호칭을 추출해 승객 세그먼트를 더 풍부하게 보기 위해 필요합니다.
# 이름에는 "Braund, Mr. Owen Harris"처럼 Mr, Mrs, Miss 같은 호칭이 들어 있습니다.

# 정규표현식 r" ([A-Za-z]+)\."은 공백 뒤에 오는 영어 단어와 마침표를 찾습니다.
title_pattern = r" ([A-Za-z]+)\."
titanic["호칭"] = titanic["Name"].str.extract(title_pattern, expand=False)

# 같은 의미로 볼 수 있는 호칭을 하나로 정리합니다.
title_replace = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
}
titanic["호칭"] = titanic["호칭"].replace(title_replace)

# 자주 나오는 호칭만 따로 남깁니다.
common_titles = ["Mr", "Miss", "Mrs", "Master"]

# 자주 나오지 않는 호칭은 Rare로 묶습니다.
rare_title_mask = ~titanic["호칭"].isin(common_titles)
titanic.loc[rare_title_mask, "호칭"] = "Rare"

titanic["호칭"].value_counts()

호칭
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

## 4. 타이타닉: 세그먼트별 생존율 막대그래프

마케팅 분석에서는 고객군별 전환율을 자주 봅니다. 타이타닉 데이터에서는 생존율을 전환율처럼 생각해 볼 수 있습니다.

### 생존율 집계하기

객실 등급과 성별 조합별로 승객 수와 평균 생존율을 계산해 막대그래프에 사용할 표를 만듭니다.


In [10]:
# 이 셀은 객실 등급과 성별 조합별 생존율을 계산하기 위해 필요합니다.
# 먼저 어떤 기준으로 그룹을 나눌지 정합니다.
segment_columns = ["객실등급", "성별"]

# 객실 등급과 성별이 같은 승객끼리 묶습니다.
segment_group = titanic.groupby(segment_columns, as_index=False)

# 각 그룹의 승객 수와 평균 생존 여부를 계산합니다.
# Survived는 생존 1, 사망 0이므로 평균을 내면 생존율이 됩니다.
titanic_segment = segment_group.agg({
    "인원": "sum",
    "Survived": "mean"
})

# 컬럼 이름을 그래프에서 읽기 쉽게 바꿉니다.
titanic_segment = titanic_segment.rename(columns={
    "인원": "승객수",
    "Survived": "생존율"
})

# 0~1 사이 값인 생존율을 0~100% 값으로 바꿉니다.
titanic_segment["생존율"] = titanic_segment["생존율"] * 100

titanic_segment

,객실등급,성별,승객수,생존율
0,1등석,남성,122,36.885246
1,1등석,여성,94,96.808511
2,2등석,남성,108,15.740741
3,2등석,여성,76,92.105263
4,3등석,남성,347,13.544669
5,3등석,여성,144,50.000000


In [11]:
# 이 셀은 객실 등급과 성별 조합별 생존율을 인터랙티브 막대그래프로 보여주기 위해 필요합니다.
# px.bar()는 Plotly Express에서 막대그래프를 만드는 함수입니다.

fig = px.bar(
    # 그래프에 사용할 데이터프레임입니다.
    titanic_segment,

    # x축에는 객실 등급을 놓습니다.
    x="객실등급",

    # y축에는 생존율을 놓습니다.
    y="생존율",

    # color는 막대 색을 나눌 기준입니다.
    # 여기서는 남성과 여성을 다른 색으로 보여줍니다.
    color="성별",

    # barmode="group"은 같은 객실 등급 안에서 남성/여성 막대를 나란히 보여줍니다.
    # "stack"으로 바꾸면 막대가 위로 쌓입니다.
    barmode="group",

    # text는 막대 위에 표시할 값입니다.
    text="생존율",

    # hover_data는 마우스를 올렸을 때 추가로 보여줄 정보입니다.
    # True는 그대로 보여주라는 뜻이고, ":.1f"는 소수점 한 자리까지 보여주라는 뜻입니다.
    hover_data={"승객수": True, "생존율": ":.1f"},

    # title은 그래프 제목입니다.
    title="객실 등급과 성별에 따른 생존율"
)

# texttemplate은 막대 위 숫자를 어떤 형식으로 보여줄지 정합니다.
# %{text:.1f}%는 text 값을 소수점 한 자리 퍼센트처럼 보여줍니다.
# textposition="outside"는 숫자를 막대 바깥쪽 위에 표시합니다.
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")

# update_layout()은 축 제목, 축 범위 같은 전체 그래프 모양을 조정합니다.
fig.update_layout(
    yaxis_title="생존율(%)",
    xaxis_title="객실 등급",
    yaxis_range=[0, 100]
)

fig

## 5. 타이타닉: 선버스트 차트

선버스트 차트는 큰 범주에서 작은 범주로 내려가며 비중을 확인할 때 유용합니다. 고객 세그먼트 구조를 한눈에 보여주는 데 자주 쓰입니다.

In [12]:
# 이 셀은 객실 등급, 성별, 생존 여부가 이어지는 구조를 선버스트 차트로 보기 위해 필요합니다.
# px.sunburst()는 큰 범주에서 작은 범주로 내려가는 원형 계층 그래프를 만듭니다.
fig = px.sunburst(
    # 그래프에 사용할 데이터프레임입니다.
    titanic,

    # path는 안쪽 원부터 바깥쪽 원까지 어떤 순서로 나눌지 정합니다.
    # 여기서는 객실 등급 -> 성별 -> 생존 여부 순서로 내려갑니다.
    path=["객실등급", "성별", "생존여부"],

    # values는 각 조각의 크기를 정하는 값입니다.
    # 인원이 모두 1이므로 승객 수를 합산하는 효과가 납니다.
    values="인원",

    # color는 색을 나눌 기준입니다.
    color="생존여부",

    # color_discrete_map은 범주별 색을 직접 지정할 때 사용합니다.
    color_discrete_map={"생존": "#2A9D8F", "사망": "#8D99AE"},
    title="타이타닉 승객 세그먼트와 생존 여부"
)

# textinfo는 조각 안에 어떤 글자를 보여줄지 정합니다.
# label은 이름, percent parent는 부모 범주 안에서 차지하는 비율입니다.
fig.update_traces(textinfo="label+percent parent")
fig

## 6. 타이타닉: 트리맵

트리맵은 각 그룹의 크기와 색을 동시에 볼 수 있습니다. 여기서는 사각형 크기로 승객 수를, 색으로 생존율을 표현합니다.

### 트리맵용 데이터 만들기

객실 등급, 성별, 연령대 조합별 승객 수와 생존율을 계산해 트리맵의 크기와 색으로 표현할 값을 준비합니다.


In [13]:
# 이 셀은 객실 등급, 성별, 연령대 조합별 승객 수와 생존율을 계산하기 위해 필요합니다.
# 트리맵은 그룹별 크기를 보여주기 때문에, 먼저 세그먼트를 더 세밀하게 나눕니다.
tree_columns = ["객실등급", "성별", "연령대"]

# 같은 객실 등급, 같은 성별, 같은 연령대에 속한 승객끼리 묶습니다.
# observed=True는 실제 데이터에 등장한 조합만 사용하겠다는 뜻입니다.
tree_group = titanic.groupby(tree_columns, observed=True, as_index=False)

# 각 세그먼트의 승객 수와 생존율을 계산합니다.
titanic_tree = tree_group.agg({
    "인원": "sum",
    "Survived": "mean"
})

# 컬럼 이름을 보기 쉽게 바꿉니다.
titanic_tree = titanic_tree.rename(columns={
    "인원": "승객수",
    "Survived": "생존율"
})

# 생존율을 퍼센트 단위로 바꿉니다.
titanic_tree["생존율"] = titanic_tree["생존율"] * 100

# 승객 수가 0인 조합은 실제로 존재하지 않는 그룹이므로 제거합니다.
has_passenger = titanic_tree["승객수"] > 0
titanic_tree = titanic_tree[has_passenger]

titanic_tree.head()

,객실등급,성별,연령대,승객수,생존율
0,1등석,남성,고령,12,8.333333
1,1등석,남성,나이 미상,21,23.809524
2,1등석,남성,어린이,3,100.000000
3,1등석,남성,중장년,54,37.037037
4,1등석,남성,청년,28,53.571429


In [14]:
# 이 셀은 승객 세그먼트의 크기와 생존율을 트리맵으로 함께 보여주기 위해 필요합니다.
# px.treemap()은 사각형 크기로 그룹의 규모를 보여주는 그래프입니다.
fig = px.treemap(
    titanic_tree,

    # path는 큰 사각형에서 작은 사각형으로 나뉘는 순서입니다.
    path=["객실등급", "성별", "연령대"],

    # values는 사각형의 크기를 정합니다.
    values="승객수",

    # color는 사각형 색을 정하는 기준입니다.
    color="생존율",

    # color_continuous_scale은 숫자값에 따라 변하는 색상 팔레트입니다.
    color_continuous_scale="Tealgrn",

    # range_color는 색상 범위를 0~100으로 고정합니다.
    # 생존율처럼 퍼센트 값일 때 해석하기 쉽습니다.
    range_color=[0, 100],

    # hover_data는 마우스를 올렸을 때 추가로 보여줄 정보입니다.
    hover_data={"승객수": True, "생존율": ":.1f"},
    title="승객 세그먼트 크기와 생존율"
)

# textinfo는 사각형 안에 이름과 값을 함께 보여주도록 설정합니다.
fig.update_traces(textinfo="label+value")
fig

## 7. 자전거 데이터 준비

자전거 데이터는 시간 흐름이 중요합니다. 날짜와 시간을 나누고, 계절과 날씨 코드를 읽기 쉬운 라벨로 바꿉니다.

### 날짜와 시간 나누기

하나의 datetime 컬럼에서 연도, 월, 시간, 요일을 분리해 시간대별 수요 패턴을 볼 수 있게 만듭니다.


In [15]:
# 이 셀은 자전거 데이터에서 날짜와 시간 변수를 만들기 위해 필요합니다.
# datetime 컬럼에는 날짜와 시간이 한꺼번에 들어 있습니다.
# 분석하기 쉽게 연도, 월, 시간, 요일을 각각 따로 꺼냅니다.

bike["year"] = bike["datetime"].dt.year
bike["month"] = bike["datetime"].dt.month
bike["hour"] = bike["datetime"].dt.hour
bike["dayofweek"] = bike["datetime"].dt.dayofweek

# 하루 단위 분석에 사용할 날짜입니다.
bike["date"] = bike["datetime"].dt.date

# 애니메이션에서 월별 장면을 만들기 위해 "2011-01" 같은 문자열을 만듭니다.
bike["year_month"] = bike["datetime"].dt.strftime("%Y-%m")

bike[["datetime", "year", "month", "hour", "dayofweek", "year_month"]].head()

,datetime,year,month,hour,dayofweek,year_month
0,2011-01-01 00:00:00,2011,1,0,5,2011-01
1,2011-01-01 01:00:00,2011,1,1,5,2011-01
2,2011-01-01 02:00:00,2011,1,2,5,2011-01
3,2011-01-01 03:00:00,2011,1,3,5,2011-01
4,2011-01-01 04:00:00,2011,1,4,5,2011-01


### 계절과 날씨 라벨 만들기

숫자 코드로 저장된 계절과 날씨를 그래프에서 읽기 쉬운 한글 라벨로 바꿉니다.


In [16]:
# 이 셀은 자전거 데이터의 코드값을 시각화용 한국어 라벨로 바꾸기 위해 필요합니다.
# 코드값 그대로 그래프에 표시하면 보는 사람이 뜻을 바로 알기 어렵습니다.

season_label = {1: "봄", 2: "여름", 3: "가을", 4: "겨울"}
bike["계절"] = bike["season"].map(season_label)

weather_label = {1: "맑음", 2: "흐림", 3: "비/눈", 4: "악천후"}
bike["날씨"] = bike["weather"].map(weather_label)

workingday_label = {0: "휴일", 1: "근무일"}
bike["근무일"] = bike["workingday"].map(workingday_label)

dayofweek_label = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
bike["요일"] = bike["dayofweek"].map(dayofweek_label)

# 8 같은 숫자보다 8시라고 표시하면 그래프에서 더 자연스럽습니다.
bike["시간대"] = bike["hour"].astype(str) + "시"

bike[["season", "계절", "weather", "날씨", "workingday", "근무일", "요일", "시간대"]].head()

,season,계절,weather,날씨,workingday,근무일,요일,시간대
0,1,봄,1,맑음,0,휴일,토,0시
1,1,봄,1,맑음,0,휴일,토,1시
2,1,봄,1,맑음,0,휴일,토,2시
3,1,봄,1,맑음,0,휴일,토,3시
4,1,봄,1,맑음,0,휴일,토,4시


## 8. 자전거: 움직이는 시간대별 수요

애니메이션은 시간이 지날수록 패턴이 어떻게 달라지는지 보여주기 좋습니다. 아래 그래프는 월별로 시간대별 평균 대여량이 어떻게 달라지는지 보여줍니다.

### 월별 시간대 수요 계산하기

월, 시간대, 근무일 여부별 평균 대여량을 계산해 애니메이션 선 그래프에 사용할 데이터를 만듭니다.


In [17]:
# 이 셀은 월, 시간대, 근무일 여부별 평균 대여량을 계산하기 위해 필요합니다.
# 애니메이션의 한 장면은 한 달입니다.
# 각 달 안에서 시간대와 근무일 여부별 평균 대여량을 계산합니다.

hour_month_columns = ["year_month", "hour", "근무일"]
hour_month_group = bike.groupby(hour_month_columns, as_index=False)
bike_hour_month = hour_month_group["count"].mean()

# 그래프에 표시될 숫자가 너무 길지 않도록 소수점 한 자리로 정리합니다.
bike_hour_month["count"] = bike_hour_month["count"].round(1)

bike_hour_month.head()

,year_month,hour,근무일,count
0,2011-01,0,근무일,8.4
1,2011-01,0,휴일,23.9
2,2011-01,1,근무일,4.1
3,2011-01,1,휴일,20.6
4,2011-01,2,근무일,1.9


In [18]:
# 이 셀은 월이 바뀔 때 시간대별 대여량 패턴이 어떻게 움직이는지 보여주기 위해 필요합니다.
# px.line()은 선 그래프를 만드는 함수입니다.
# animation_frame에 year_month를 넣으면 월별로 재생되는 그래프가 만들어집니다.
max_count = bike_hour_month["count"].max()

fig = px.line(
    bike_hour_month,

    # x축에는 하루 24시간을 놓습니다.
    x="hour",

    # y축에는 평균 대여량을 놓습니다.
    y="count",

    # color는 선을 나눌 기준입니다. 근무일과 휴일이 다른 색으로 그려집니다.
    color="근무일",

    # animation_frame은 재생 버튼을 눌렀을 때 바뀌는 장면 기준입니다.
    animation_frame="year_month",

    # markers=True는 선 위에 점도 함께 표시합니다.
    markers=True,

    # range_x, range_y는 축 범위를 고정합니다.
    # 애니메이션이 재생될 때 축이 계속 바뀌지 않도록 하기 위해 사용합니다.
    range_x=[0, 23],
    range_y=[0, max_count * 1.1],
    title="월별 시간대 수요 변화"
)
fig.update_layout(xaxis_title="시간", yaxis_title="평균 대여량")
fig

## 9. 자전거: 움직이는 날씨 버블 차트

버블 차트는 여러 정보를 한 번에 보여줄 수 있습니다. 여기서는 기온, 습도, 대여량, 계절, 월 변화를 함께 봅니다.

### 버블 차트용 평균값 계산하기

월, 시간, 계절별 평균 대여량과 평균 기온, 평균 습도를 계산해 버블 차트에 사용할 값을 준비합니다.


In [19]:
# 이 셀은 움직이는 버블 차트에 사용할 월, 시간, 계절별 평균 값을 계산하기 위해 필요합니다.
# 버블 차트에는 평균 대여량, 평균 기온, 평균 습도처럼 여러 숫자가 함께 들어갑니다.

# 먼저 어떤 기준으로 데이터를 묶을지 정합니다.
bubble_columns = ["year_month", "hour", "계절"]
bubble_group = bike.groupby(bubble_columns, as_index=False)

# count, temp, humidity의 평균을 계산합니다.
bike_bubble = bubble_group[["count", "temp", "humidity"]].mean()

# 그래프에서 읽기 쉬운 컬럼 이름으로 바꿉니다.
bike_bubble = bike_bubble.rename(columns={
    "count": "평균대여량",
    "temp": "평균기온",
    "humidity": "평균습도"
})

# 마우스를 올렸을 때 8시, 9시처럼 보이게 시간대 라벨을 만듭니다.
bike_bubble["시간대"] = bike_bubble["hour"].astype(str) + "시"

bike_bubble.head()

,year_month,hour,계절,평균대여량,평균기온,평균습도,시간대
0,2011-01,0,봄,14.388889,7.926667,61.055556,0시
1,2011-01,1,봄,10.500000,7.562222,62.888889,1시
2,2011-01,2,봄,7.235294,7.380000,64.529412,2시
3,2011-01,3,봄,6.000000,7.790000,59.500000,3시
4,2011-01,4,봄,2.066667,7.981333,61.533333,4시


In [20]:
# 이 셀은 기온, 습도, 대여량의 관계가 월별로 어떻게 움직이는지 보여주기 위해 필요합니다.
# px.scatter()는 산점도를 만드는 함수입니다.
# x축과 y축 범위를 미리 정해두면 애니메이션이 재생될 때 화면이 흔들리지 않습니다.
x_min = bike_bubble["평균기온"].min() - 2
x_max = bike_bubble["평균기온"].max() + 2
y_min = bike_bubble["평균습도"].min() - 5
y_max = bike_bubble["평균습도"].max() + 5

fig = px.scatter(
    bike_bubble,

    # x축에는 평균 기온을 놓습니다.
    x="평균기온",

    # y축에는 평균 습도를 놓습니다.
    y="평균습도",

    # size는 점의 크기를 정하는 기준입니다.
    # 평균 대여량이 클수록 점이 커집니다.
    size="평균대여량",

    # color는 점의 색을 나눌 기준입니다.
    color="계절",

    # animation_frame은 월별로 장면이 바뀌게 만듭니다.
    animation_frame="year_month",

    # animation_group은 애니메이션에서 같은 대상을 이어서 추적할 기준입니다.
    # 여기서는 같은 시간대의 점이 월별로 움직이는 것처럼 보이게 합니다.
    animation_group="시간대",

    # hover_name은 마우스를 올렸을 때 가장 크게 보여줄 이름입니다.
    hover_name="시간대",

    # size_max는 가장 큰 점의 최대 크기입니다.
    size_max=45,

    # range_x와 range_y는 애니메이션 중 축 범위를 고정합니다.
    range_x=[x_min, x_max],
    range_y=[y_min, y_max],
    title="기온과 습도 속에서 움직이는 자전거 대여 수요"
)
fig.update_layout(xaxis_title="평균 기온", yaxis_title="평균 습도")
fig

## 10. 자전거: 요일과 시간대 히트맵

히트맵은 “언제 수요가 높은가”를 빠르게 찾을 때 좋습니다. 운영 기획에서는 인력 배치, 재고 배치, 프로모션 시간대를 정할 때 이런 형태를 자주 사용합니다.

### 히트맵용 표 만들기

요일을 행으로, 시간을 열로 두고 평균 대여량을 채워 히트맵에 바로 넣을 수 있는 표를 만듭니다.


In [21]:
# 이 셀은 요일과 시간대별 평균 대여량을 표 형태로 만들기 위해 필요합니다.
# 히트맵은 행과 열로 된 표 데이터를 색으로 표현합니다.
# 여기서는 행은 요일, 열은 시간, 값은 평균 대여량입니다.

bike_heatmap = pd.pivot_table(
    bike,
    values="count",
    index="요일",
    columns="hour",
    aggfunc="mean"
)

# 요일이 월요일부터 일요일 순서로 보이도록 행 순서를 정리합니다.
day_order = ["월", "화", "수", "목", "금", "토", "일"]
bike_heatmap = bike_heatmap.reindex(day_order)

bike_heatmap.round(1)

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
요일,,,,,,,,,,,,,,,,,,,,,
월,35.5,18.1,10.7,5.7,6.1,22.4,89.2,260.4,428.1,226.4,...,199.6,210.2,295.0,521.4,499.6,359.9,249.7,179.0,120.0,66.8
화,27.3,11.9,6.2,4.0,5.2,24.0,105.4,297.6,469.2,236.1,...,162.5,181.1,284.1,544.2,522.8,356.1,249.9,183.2,130.6,76.1
수,36.2,15.6,8.4,5.0,4.6,25.0,105.8,297.2,485.2,238.8,...,166.9,179.6,268.4,509.3,489.4,348.2,251.6,190.7,140.4,80.1
목,37.5,15.4,8.4,4.9,5.3,25.5,108.2,307.7,496.6,241.8,...,174.7,193.5,288.3,536.6,506.8,364.0,272.3,199.7,148.9,99.6
금,53.2,24.5,12.5,6.3,5.9,23.4,91.4,254.1,470.2,262.4,...,227.1,248.5,331.5,502.0,427.6,303.6,216.8,170.9,152.6,119.5
토,98.2,70.0,50.3,23.1,7.7,8.5,21.1,47.2,117.6,190.6,...,398.4,398.8,380.2,346.3,300.7,250.0,188.1,159.0,143.4,120.0
일,96.2,79.5,62.5,30.4,9.7,9.5,15.1,34.7,84.0,158.7,...,370.4,364.5,365.5,326.3,273.5,227.3,172.6,131.9,99.8,64.8


In [22]:
# 이 셀은 요일과 시간대별 수요 강도를 히트맵으로 보여주기 위해 필요합니다.
# px.imshow()는 표 형태의 숫자를 색으로 보여주는 히트맵 함수입니다.
fig = px.imshow(
    # 행과 열로 정리된 평균 대여량 표를 넣습니다.
    bike_heatmap,

    # color_continuous_scale은 숫자가 작고 클 때 어떤 색으로 보일지 정합니다.
    color_continuous_scale="YlOrRd",

    # aspect="auto"는 화면 크기에 맞게 칸의 가로세로 비율을 자동 조정합니다.
    aspect="auto",
    title="요일과 시간대별 평균 대여량 히트맵"
)
fig.update_layout(xaxis_title="시간", yaxis_title="요일")
fig

## 11. 자전거: 3D 산점도

3D 산점도는 발표나 탐색 단계에서 흥미를 끌기 좋습니다. 다만 정확한 비교에는 2D 그래프나 표가 더 적합할 수 있으므로, “탐색용”으로 사용하는 것이 좋습니다.

### 3D 산점도용 샘플 만들기

전체 데이터를 모두 그리면 그래프가 무거워질 수 있으므로 일부 행만 뽑아 3D 산점도에 사용합니다.


In [23]:
# 이 셀은 3D 산점도가 너무 무거워지지 않도록 일부 행만 샘플링하기 위해 필요합니다.
bike_sample = bike.sample(2000, random_state=42)
bike_sample.shape

(2000, 23)

In [24]:
# 이 셀은 기온, 습도, 대여량의 관계를 3D 공간에서 살펴보기 위해 필요합니다.
# px.scatter_3d()는 3차원 산점도를 만드는 함수입니다.
fig = px.scatter_3d(
    bike_sample,

    # x, y, z는 3D 공간의 세 축에 들어갈 변수입니다.
    x="temp",
    y="humidity",
    z="count",

    # color는 시간대별로 점 색을 다르게 보여줍니다.
    color="hour",

    # size는 점의 크기를 정합니다. 대여량이 많을수록 큰 점으로 보입니다.
    size="count",

    # opacity는 점의 투명도입니다. 점이 겹칠 때 덜 답답하게 보입니다.
    opacity=0.55,

    # hover_data는 마우스를 올렸을 때 추가로 보여줄 정보입니다.
    hover_data=["datetime", "계절", "근무일"],
    title="기온, 습도, 대여량의 3D 관계"
)
fig.update_layout(scene=dict(
    xaxis_title="기온",
    yaxis_title="습도",
    zaxis_title="대여량"
))
fig

## 12. 시각화 선택 기준

그래프가 화려하다고 항상 좋은 것은 아닙니다. 목적에 맞는 그래프를 고르는 것이 더 중요합니다.

| 목적 | 추천 시각화 |
|---|---|
| 세그먼트별 성과 비교 | 막대그래프, 트리맵 |
| 시간에 따른 패턴 변화 | 애니메이션 선 그래프 |
| 두 변수와 크기를 동시에 표현 | 버블 차트 |
| 시간대별 운영 포인트 찾기 | 히트맵 |
| 발표에서 탐색 느낌 주기 | 3D 산점도 |

좋은 시각화는 예쁜 그림에서 끝나지 않습니다. 어떤 의사결정을 도와주는지까지 함께 말할 수 있어야 합니다.